# conv-leakyrelu-block-discriminator — ex2: stride=1 Conv+BN+LeakyReLU block — same H/W, deeper features

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-leakyrelu-block-discriminator`. Running the final beacon cell reports progress against the `GAN: Conv+LeakyReLU discriminator block` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Conv+LeakyReLU discriminator block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-leakyrelu-block-discriminator`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-leakyrelu-block-discriminator"
DD_SUBTOPIC = "GAN: Conv+LeakyReLU discriminator block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv2d + BN + LeakyReLU — stride=1 (no downsample) variant

Ex1 built a `Conv2d(stride=2) → BN → LeakyReLU` block — DCGAN's down-sampling discriminator unit. The deepening move keeps the topology but flips `stride=1` so the spatial dimensions are PRESERVED — same kernel size + padding chosen so H_out == H_in.

**Spatial formula:** `H_out = floor((H_in + 2*pad - kernel) / stride) + 1`. 
With `kernel=3, stride=1, pad=1`: `H_out = floor((H + 2 - 3) / 1) + 1 = H`. The kernel CAN reach every spatial position; padding picks up the boundary slack.

```python
block = nn.Sequential(
    nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
    nn.BatchNorm2d(out_ch),
    nn.LeakyReLU(0.2, inplace=True),
)
```

**Why this variant matters.** DCGAN discriminators alternate stride-2 (downsample) and stride-1 (refine) blocks. The stride-1 block is where you ADD CAPACITY without changing the feature-map size — channel growth without spatial collapse.

**`bias=False` still applies.** BatchNorm has its own affine bias; the conv's bias would be immediately subtracted out by BN's mean-centering. Save the parameters.

### Exercise 2 — stride=1 Conv+BN+LeakyReLU block — same H/W, deeper features

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `Conv2d(stride=1, padding=1, kernel_size=3, bias=False)` followed by BatchNorm2d + LeakyReLU(0.2) to build a spatial-preserving refinement block whose output has the same H/W as its input.
> Keywords: conv, stride1, leaky-relu, discriminator
> ```

**KCs targeted:** `stride1-padding-for-spatial-preservation`, `conv-bn-leakyrelu-bias-false`

Build `ex2_build_stride1_block(in_ch, out_ch)`. The spatial-preserving variant of ex1's stride=2 downsample block.

Constraints:
1. Return an `nn.Sequential` with three children in order:
   - `nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)`
   - `nn.BatchNorm2d(out_ch)`
   - `nn.LeakyReLU(0.2, inplace=True)`
2. The output spatial dims must EQUAL the input spatial dims for any H, W >= 3 (verified by the test).
3. `bias=False` on the Conv — BatchNorm has its own affine bias.
4. LeakyReLU slope = 0.2, `inplace=True` (the DCGAN convention).

Output: an `nn.Sequential` callable as `block(x)` where `x: (B, in_ch, H, W)` → `out: (B, out_ch, H, W)`.

In [ ]:
def ex2_build_stride1_block(in_ch: int, out_ch: int) -> nn.Module:
    """Spatial-preserving Conv+BN+LeakyReLU block (stride=1, padding=1)."""
    raise NotImplementedError()


def _test_ex2():
    # === Build and inspect the block structure ===
    block = ex2_build_stride1_block(in_ch=8, out_ch=16)
    assert isinstance(block, nn.Sequential), f'must return nn.Sequential, got {type(block).__name__}'
    children = list(block.children())
    assert len(children) == 3, f'must have exactly 3 children, got {len(children)}'
    assert isinstance(children[0], nn.Conv2d), f'child 0 must be Conv2d, got {type(children[0]).__name__}'
    assert isinstance(children[1], nn.BatchNorm2d), f'child 1 must be BatchNorm2d, got {type(children[1]).__name__}'
    assert isinstance(children[2], nn.LeakyReLU), f'child 2 must be LeakyReLU, got {type(children[2]).__name__}'

    # === Conv hyperparams ===
    conv = children[0]
    assert conv.in_channels == 8 and conv.out_channels == 16
    assert conv.kernel_size == (3, 3), f'kernel_size must be 3, got {conv.kernel_size}'
    assert conv.stride == (1, 1), f'stride must be 1 (spatial-preserving), got {conv.stride}'
    assert conv.padding == (1, 1), f'padding must be 1, got {conv.padding}'
    assert conv.bias is None, 'bias must be False (BN follows)'

    # === BN matches Conv out_channels ===
    bn = children[1]
    assert bn.num_features == 16, f'BN num_features must match conv out_channels, got {bn.num_features}'

    # === LeakyReLU slope + inplace ===
    act = children[2]
    assert act.negative_slope == 0.2, f'slope must be 0.2, got {act.negative_slope}'
    assert act.inplace is True, 'LeakyReLU must be inplace=True'

    # === Output H/W matches input H/W (spatial preservation) ===
    for H, W in [(4, 4), (8, 8), (7, 11), (32, 32), (3, 3)]:
        x = t.randn(2, 8, H, W)
        out = block(x)
        assert out.shape == (2, 16, H, W), f'spatial preservation failed for {(H, W)}: got {tuple(out.shape)}'

    # === Different (in_ch, out_ch) combos ===
    block2 = ex2_build_stride1_block(in_ch=3, out_ch=32)
    x = t.randn(1, 3, 16, 16)
    out = block2(x)
    assert out.shape == (1, 32, 16, 16), f'channel growth path wrong: {tuple(out.shape)}'

    # === Same in_ch == out_ch (refinement block, no channel change) ===
    block3 = ex2_build_stride1_block(in_ch=64, out_ch=64)
    x = t.randn(2, 64, 8, 8)
    out = block3(x)
    assert out.shape == (2, 64, 8, 8)

    # === Output is nonnegative on positive inputs ===
    block4 = ex2_build_stride1_block(in_ch=4, out_ch=4)
    # Force the block deterministically — eval mode freezes BN.
    block4.eval()
    x = t.randn(2, 4, 6, 6) * 10  # large magnitude → BN doesn't crush; LeakyReLU still leaks negatives
    out = block4(x)
    # Negative outputs still possible (slope 0.2), but magnitudes scaled toward small.
    # Check that the LeakyReLU slope is at work: for any x_after_bn < 0, output = 0.2 * x_after_bn,
    # so the ratio output_neg / x_after_bn_neg should be ~0.2.
    # We just sanity-check shape + finiteness here.
    assert out.shape == (2, 4, 6, 6)
    assert t.isfinite(out).all()
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_build_stride1_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.LeakyReLU(0.2, inplace=True),
    )
```

**`kernel=3, stride=1, padding=1` is the canonical spatial-preserving conv.** The 'SAME' padding pattern from TF. With `H_out = floor((H + 2*1 - 3) / 1) + 1 = H`, the kernel can reach every position and boundary slack is taken up by the 1-pad.

**Stride-1 refinement blocks are where DCGAN puts CAPACITY.** Stride-2 blocks halve the spatial resolution AND grow channels. Stride-1 blocks keep the spatial size and either grow channels (input-to-internal layer) or hold them constant (pure refinement). They cost 4× the FLOPs of a stride-2 block at the same spatial size, so use them sparingly.

**`bias=False` is a habitual save.** A trainable bias before BN gets immediately subtracted out by BN's mean-centering — the optimizer would still train it, but its effect on the output is zero. Save the parameters.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()